# Libary dan import



In [3]:
from ortools.sat.python import cp_model
import torch
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_add_pool
from torch_geometric.data import Data
from torch.distributions import Categorical
from torch_geometric.loader import DataLoader
from types import SimpleNamespace
import networkx as nx
import numpy as np
import gymnasium as gym
import random
import os  # Tetap pakai os untuk manajemen folder lokal
from torch.utils.tensorboard import SummaryWriter
from gymnasium import spaces

# Library JSSP
from job_shop_lib.generation import (
    modular_instance_generator,
    get_default_machine_matrix_creator,
    get_default_duration_matrix_creator,
    range_size_selector
)

base_path = r"D:\tugas akhir"
print(f"Bekerja di folder: {base_path}")

load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\zlib1.dll...
load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\abseil_dll.dll...
load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\utf8_validity.dll...
load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\re2.dll...
load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\libprotobuf.dll...
load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\highs.dll...
load d:\tugas akhir\venv_skripsi\lib\site-packages\ortools\.libs\ortools.dll...



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "d:\tugas akhir\venv_skripsi\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\tugas akhir\venv_skripsi\lib\site-packages\traitlets\config\application.py", line 1082, in launch_instan

Bekerja di folder: D:\tugas akhir


#Class program

##Disjungtif Grapf

In [4]:
class DisjunctiveGraph:
    def __init__(self, instance):
        # 1. Ambil data dari instance
        self.num_jobs = instance.num_jobs
        self.num_machines = instance.num_machines
        self.total_ops = instance.num_operations

        # Mapping untuk mempermudah pelacakan (Node ID -> Job ID, Step)
        self.node_to_job = {}

        self.duration_matrix = np.array(instance.duration_matrix)
        self.machine_matrix = np.array(instance.machines_matrix)

        # 2. Konstanta Normalisasi
        self.max_duration = instance.max_duration if instance.max_duration > 0 else 1.0

        # 3. Fitur Node (Tensor 8 dimensi)
        self.node_features = torch.zeros((self.total_ops, 8), dtype=torch.float)

        # 4. State Pelacakan Waktu
        self.arrival_times = np.full(self.total_ops, -1.0)
        self.start_times = np.full(self.total_ops, -1.0)
        self.finish_times = np.full(self.total_ops, -1.0)

        # 5. Struktur Graf
        self.nx_graph = nx.DiGraph()
        self.ops_in_job = {j: [] for j in range(self.num_jobs)}
        self.ops_in_machine = {m: [] for m in range(self.num_machines)}

        self._build_graph()

    def _build_graph(self):
        global_id = 0
        for j in range(self.num_jobs):
            total_job_time = np.sum(self.duration_matrix[j])

            for step in range(self.num_machines):
                duration = self.duration_matrix[j, step]
                machine_id = self.machine_matrix[j, step]

                # Simpan mapping
                self.node_to_job[global_id] = (j, step)
                self.ops_in_job[j].append(global_id)
                self.ops_in_machine[machine_id].append(global_id)

                # Inisialisasi 8 Fitur
                self.node_features[global_id, 0] = 1.0  # Unscheduled
                self.node_features[global_id, 3] = float(duration) / self.max_duration
                self.node_features[global_id, 4] = (duration / total_job_time) if total_job_time > 0 else 0
                self.node_features[global_id, 5] = ((self.num_machines - 1) - step) / self.num_machines

                self.nx_graph.add_node(global_id)

                # Conjunctive Edge (Hanya searah di NetworkX)
                if step > 0:
                    self.nx_graph.add_edge(global_id - 1, global_id, type='conjunctive')

                global_id += 1

        # Disjunctive Edges (Hanya satu arah di NetworkX untuk efisiensi)
        for m, ops in self.ops_in_machine.items():
            for i in range(len(ops)):
                for k in range(i + 1, len(ops)):
                    self.nx_graph.add_edge(ops[i], ops[k], type='disjunctive')

        # Waktu kedatangan awal untuk operasi pertama setiap job
        for j in range(self.num_jobs):
            first_op = self.ops_in_job[j][0]
            self.arrival_times[first_op] = 0.0

    def update_graph_state(self, current_time):
        for node_id in range(self.total_ops):
            is_unscheduled = (self.node_features[node_id, 0] == 1.0)
            is_processing = (self.node_features[node_id, 1] == 1.0)

            # (e) Waktu Tunggu
            arrival = self.arrival_times[node_id]
            if is_unscheduled and arrival != -1 and current_time >= arrival:
                wait_duration = current_time - arrival
                self.node_features[node_id, 6] = float(wait_duration) / self.max_duration

            # (f) Sisa Waktu
            if is_processing:
                remaining = max(0, self.finish_times[node_id] - current_time)
                self.node_features[node_id, 7] = float(remaining) / self.max_duration
            else:
                self.node_features[node_id, 7] = 0.0

    def mark_operation_start(self, node_id, current_time):
        # Gunakan durasi dari matriks aslinya
        j, step = self.node_to_job[node_id]
        duration = self.duration_matrix[j, step]

        self.node_features[node_id, 0] = 0.0
        self.node_features[node_id, 1] = 1.0 # Processing
        self.node_features[node_id, 2] = 0.0

        self.start_times[node_id] = current_time
        self.finish_times[node_id] = current_time + duration
        self.node_features[node_id, 7] = float(duration) / self.max_duration

    def mark_operation_finish(self, node_id, current_time):
        self.node_features[node_id, 0] = 0.0
        self.node_features[node_id, 1] = 0.0
        self.node_features[node_id, 2] = 1.0 # Completed
        self.node_features[node_id, 7] = 0.0

        j, step = self.node_to_job[node_id]
        if step < self.num_machines - 1:
            next_node = self.ops_in_job[j][step + 1]
            self.arrival_times[next_node] = current_time

    def get_graph_data(self):
        x = self.node_features.clone()
        edge_p, edge_s, edge_d = [], [], []

        for u, v, data in self.nx_graph.edges(data=True):
            if data['type'] == 'conjunctive':
                edge_p.append([u, v])
                edge_s.append([v, u])
            elif data['type'] == 'disjunctive':
                edge_d.append([u, v])
                edge_d.append([v, u])

        # Fungsi pembantu untuk membuat tensor aman (mengatasi list kosong)
        def to_tensor(edge_list):
            if len(edge_list) > 0:
                return torch.tensor(edge_list, dtype=torch.long).t().contiguous()
            else:
                return torch.empty((2, 0), dtype=torch.long)

        return Data(
            x=x,
            edge_index_p=to_tensor(edge_p),
            edge_index_s=to_tensor(edge_s),
            edge_index_d=to_tensor(edge_d)
        )

##Enviroment

In [5]:
class JSSPEnv(gym.Env):
    def __init__(self, generator):
        super(JSSPEnv, self).__init__()
        self.generator = generator
        self.current_time = 0.0
        self.graph = None
        # Action space disesuaikan dengan total operasi maksimal (misal 1000)
        self.action_space = spaces.Discrete(1000)

    def _get_observations(self):
        # Sekarang mengembalikan objek Data utuh sesuai revisi kita sebelumnya
        return self.graph.get_graph_data()

    def get_legal_actions(self):
        legal_actions = []

        # 1. Identifikasi mesin yang sedang sibuk
        busy_machines = set()
        for node_id in range(self.graph.total_ops):
            if self.graph.node_features[node_id, 1] == 1.0: # is_processing
                # Kita cari mesinnya dari duration_matrix/machine_matrix
                job_id, step = self.graph.node_to_job[node_id]
                m_id = self.graph.machine_matrix[job_id, step]
                busy_machines.add(m_id)

        # 2. Cari node yang siap dikerjakan
        for node_id in range(self.graph.total_ops):
            is_unscheduled = (self.graph.node_features[node_id, 0] == 1.0)

            # Cek mesin untuk node ini
            j_id, s_id = self.graph.node_to_job[node_id]
            m_id = self.graph.machine_matrix[j_id, s_id]
            machine_is_free = (m_id not in busy_machines)

            # Cet Precedence (apakah operasi sebelumnya dalam job yang sama sudah selesai)
            predecessor_done = True
            if s_id > 0:
                prev_node = self.graph.ops_in_job[j_id][s_id - 1]
                if self.graph.node_features[prev_node, 2] != 1.0: # index 2: completed
                    predecessor_done = False

            if is_unscheduled and machine_is_free and predecessor_done:
                legal_actions.append(node_id)

        return legal_actions # Indentasi diperbaiki (di luar loop for)

    def _advance_time(self):
        """Mekanisme lompat waktu (Event-based)"""
        while True:
            legal_actions = self.get_legal_actions()
            done = self._check_if_done()

            # Berhenti jika ada yang bisa dikerjakan atau semua tamat
            if len(legal_actions) > 0 or done:
                break

            # Jika macet (tidak ada legal action tapi belum done), lompat ke finish time terdekat
            active_finish_times = [
                self.graph.finish_times[i]
                for i in range(self.graph.total_ops)
                if self.graph.node_features[i, 1] == 1.0
            ]

            if active_finish_times:
                self.current_time = min(active_finish_times)
                # Tandai semua yang selesai pada waktu tersebut
                for i in range(self.graph.total_ops):
                    if self.graph.node_features[i, 1] == 1.0 and self.graph.finish_times[i] <= self.current_time:
                        # Tambahkan self.current_time sebagai argumen sesuai method di DisjunctiveGraph
                        self.graph.mark_operation_finish(i, self.current_time)

                self.graph.update_graph_state(self.current_time)
            else:
                break

    def step(self, action):
        # 1. Jalankan aksi
        self.graph.mark_operation_start(action, self.current_time)

        # 2. Majukan waktu jika perlu (sampai ada aksi legal baru tersedia)
        self._advance_time()

        # 3. Update kondisi fitur
        self.graph.update_graph_state(self.current_time)

        # 4. REWARD SESUAI PAPER: Negatif dari jumlah "Waiting Jobs"
        waiting_jobs = 0
        for j in range(self.graph.num_jobs):
            # Ambil ID operasi terakhir dari job ke-j
            last_op_id = self.graph.ops_in_job[j][-1]

            # Jika operasi terakhir belum selesai, berarti job ini masih "menunggu/berjalan"
            if self.graph.node_features[last_op_id, 2] != 1.0:
                waiting_jobs += 1

        reward = -float(waiting_jobs)

        done = self._check_if_done()
        obs = self._get_observations()

        return obs, reward, done, False, {}

    def reset(self, seed=None, options=None, instance=None):
        super().reset(seed=seed)

        # 1. Jika instance diberikan (untuk Validasi atau Batch Training), gunakan itu.
        # Jika tidak, ambil dari generator (untuk Training biasa).
        if instance is not None:
            selected_instance = instance
        else:
            selected_instance = next(self.generator)

        # 2. Buat Graf Disjungtif baru dari instance terpilih
        self.graph = DisjunctiveGraph(selected_instance)

        # 3. Reset waktu simulasi
        self.current_time = 0.0

        # 4. Ambil observasi awal
        obs = self._get_observations()

        return obs, {}

    def _check_if_done(self):
        # Jika semua node fitur index 2 (Completed) bernilai 1.0, maka selesai
        # Kita gunakan torch.all karena self.graph.node_features adalah tensor
        return torch.all(self.graph.node_features[:, 2] == 1.0).item()

    def generate_instance(self, jumlah):
        return [next(self.generator) for _ in range(jumlah)]

##Message passing arsitektur

In [6]:
class JSSP_Conv_Jalur(MessagePassing):
    """Implementasi Agregasi Lokal Terpisah (Poin a)"""
    def __init__(self, in_dim, out_dim):
        # Sigma sesuai rumus agregasi di paper
        super(JSSP_Conv_Jalur, self).__init__(aggr='add')
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 256), # Sesuai ukuran di paper
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, out_dim)
        )

    def forward(self, x, edge_index):
        # 1. Agregasi tetangga (Message Passing)
        aggr_out = self.propagate(edge_index, x=x)
        # 2. Aplikasi fungsi F (MLP) setelah agregasi
        return self.mlp(aggr_out)

##GNN model

In [7]:
class JSSPGNN(nn.Module):
    def __init__(self, num_node_features=8):
        super(JSSPGNN, self).__init__()

        # (a) Agregasi Lokal Terpisah (Fp, Fs, Fd)
        self.Fp = JSSP_Conv_Jalur(8, 8)
        self.Fs = JSSP_Conv_Jalur(8, 8)
        self.Fd = JSSP_Conv_Jalur(8, 8)

        # (d) Fusi Akhir (Fn) - Menerima 48 dimensi (8*6 kombinasi fitur)
        self.Fn = nn.Sequential(
            nn.Linear(48, 256), # Tetap 256 sesuai paper
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 8) # Kembali ke dimensi embedding 8
        )

        # Layer untuk Policy (Actor) dan Value (Critic)
        self.actor_layer = nn.Linear(8, 1)
        self.critic_layer = nn.Linear(8, 1)

    def forward(self, data):
        # Mengambil data dari objek Data PyG (Hasil dari JSSPEnv)
        x = data.x

        edge_index_p = data.edge_index_p
        edge_index_s = data.edge_index_s
        edge_index_d = data.edge_index_d

        # Penanda untuk operasi yang belum selesai
        not_finished_mask = (x[:, 2] == 0).float().unsqueeze(-1)

       # 1. PASTIKAN BATCH ADALAH 1D (Penting!)
        if hasattr(data, 'batch') and data.batch is not None:
            batch = data.batch.view(-1)
        else:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        h_initial = x * not_finished_mask # h(0)v - Fitur statis/awal
        h_current = x * not_finished_mask # h(k-1)v

        # PROSES ITERASI (3 KALI SESUAI RUMUS PAPER)
        for k in range(3):
            # (a) Agregasi Lokal Jalur Terpisah
            out_p = F.relu(self.Fp(h_current, edge_index_p)) * not_finished_mask
            out_s = F.relu(self.Fs(h_current, edge_index_s)) * not_finished_mask
            out_d = F.relu(self.Fd(h_current, edge_index_d)) * not_finished_mask

            # (b) Global Pooling (Status Global Pabrik)
            # Menggunakan batch agar informasi global tidak bercampur antar graf
            sum_v = global_add_pool(h_current, batch)

            # Broadcast kembali informasi global ke setiap node dalam batch yang sama
            out_global = sum_v[batch] * not_finished_mask

            # Debugging untuk memastikan semua 2D
            # print(f"p: {out_p.shape}, s: {out_s.shape}, d: {out_d.shape}, global: {out_global.shape}, cur: {h_current.shape}, ini: {h_initial.shape}")

            # (c) & (d) Fusi Konkat 48 Dimensi
            # Urutan: p, s, d, global, current, initial
            fusi_48 = torch.cat([
                out_p, out_s, out_d, out_global, h_current, h_initial
            ], dim=-1)

            # Update h_current untuk iterasi k berikutnya
            h_current = self.Fn(fusi_48) * not_finished_mask

        # Output Actor: Logits untuk tiap node (untuk pemilihan aksi)
        logits = self.actor_layer(h_current).squeeze(-1)

        # Output Critic: Value tunggal untuk satu state graf
        # Mengambil rata-rata/pool dari semua embedding node untuk representasi graf
        graph_repr = global_add_pool(h_current, batch)
        value = self.critic_layer(graph_repr)

        return logits, value

##Buffer memory

In [8]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.is_terminals = []
        self.legal_masks = []      # Menyimpan mask agar tidak perlu dihitung ulang saat PPO update

    def clear(self):
        del self.states[:]
        del self.actions[:]
        del self.log_probs[:]
        del self.rewards[:]
        del self.values[:]
        del self.is_terminals[:]
        del self.legal_masks[:]

    # PERBAIKAN: Tambahkan legal_mask di parameter dan append
    def push(self, state, action, log_prob, reward, value, is_terminal, legal_mask):
        self.states.append(state)
        self.actions.append(torch.tensor(action))
        self.log_probs.append(log_prob)
        self.rewards.append(torch.tensor(reward))
        self.values.append(value)
        self.is_terminals.append(torch.tensor(is_terminal))
        self.legal_masks.append(legal_mask)

##PPO update pembelajaran

In [ ]:
class PPO:
    def __init__(self, model,device, lr=2.5e-4, gamma=1.0, K_epochs=4, eps_clip=0.2):
        self.device = device        # untuk penggunaan GPU
        self.gamma = gamma          # Discount factor (sesuai tabel: 1.0)
        self.eps_clip = eps_clip    # Clipping parameter (0.2)
        self.K_epochs = K_epochs    # Epochs (4)
        self.gae_lambda = 0.95      # Lambda GAE
        self.v_coeff = 0.5          # Value coefficient (alpha)
        self.ent_coeff = 0.1       # Entropy coefficient (beta)

        self.policy = model
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr)

        # Policy Old untuk menghitung rasio r_t(theta)
        self.policy_old = copy.deepcopy(model).to(device)

        self.policy_old.load_state_dict(self.policy.state_dict())

        self.MseLoss = nn.MSELoss()

    def select_action(self, state, legal_actions):
        with torch.no_grad():
            logits, value = self.policy_old(state)

            # 1. Masking Ilegal Actions
            mask = torch.full(logits.shape, -1e9).to(logits.device)
            mask[legal_actions] = 0
            masked_logits = logits + mask

            # 2. Distribusi Probabilitas (PERBAIKAN STABILITAS)
            # Biarkan PyTorch yang menghitung softmax secara internal
            dist = Categorical(logits=masked_logits)

            action = dist.sample()
            action_logprob = dist.log_prob(action)

        return action.item(), action_logprob.item(), value.item(), mask

    def update(self, buffer):
        self.policy.train()
        # --- 1. PERHITUNGAN GAE & TARGET RETURNS ---
        rewards_target = []
        advantages = []
        gae = 0

        # Tambahkan 0 untuk state terminal masa depan
        values_list = buffer.values + [0]

        for i in reversed(range(len(buffer.rewards))):
            # mask=0 jika terminal, mask=1 jika tidak
            mask = 1.0 - buffer.is_terminals[i].float()

            # delta_t = R_t + gamma * V(s_t+1) - V(s_t)
            delta = buffer.rewards[i] + (self.gamma * values_list[i+1] * mask) - values_list[i]

            # A_hat_t = delta_t + (gamma * lambda * A_hat_t+1)
            gae = delta + (self.gamma * self.gae_lambda * gae * mask)

            advantages.insert(0, gae)
            # V_target = A_hat_t + V(s_t)
            rewards_target.insert(0, gae + values_list[i])

        # Konversi ke tensor
        advantages = torch.tensor(advantages, dtype=torch.float32).to(self.device)
        rewards_target = torch.tensor(rewards_target, dtype=torch.float32).to(self.device)
        old_actions = torch.tensor(buffer.actions).to(self.device)
        old_logprobs = torch.tensor(buffer.log_probs).to(self.device)
        old_masks = torch.cat(buffer.legal_masks, dim=0).to(self.device)

        # Normalisasi Advantage (Penting untuk stabilitas)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-7)

        # Menggunakan DataLoader untuk batching objek Data PyG
        loader = DataLoader(buffer.states, batch_size=len(buffer.states))
        batch_state = next(iter(loader))
        batch_state = batch_state.to(self.device)

        # --- 2. OPTIMASI K-EPOCHS ---
        for _ in range(self.K_epochs):
            # Evaluasi state dengan policy saat ini
            logits, values = self.policy(batch_state)
            values = values.squeeze(-1) # Pastikan shape [N] sama dengan rewards_target

            # Masking ulang logits untuk menghitung log_prob baru
            # old_masks.view(-1) meratakan mask jika menggunakan batching
            masked_logits = logits + old_masks.view(-1)
            dist = Categorical(logits=masked_logits)

            logprobs = dist.log_prob(old_actions)
            dist_entropy = dist.entropy()

            # Rasio r_t(theta) = pi_new / pi_old
            ratios = torch.exp(logprobs - old_logprobs)

            # Clipped Surrogate Loss
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages

            loss_actor = -torch.min(surr1, surr2).mean()

            # Value Loss (MSE) - Persamaan (10)
            loss_critic = self.v_coeff * self.MseLoss(values, rewards_target)

            # Entropy Loss (Mendorong Eksplorasi)
            loss_entropy = -self.ent_coeff * dist_entropy.mean()

            # Total Loss
            total_loss = loss_actor + loss_critic + loss_entropy

            # Gradient Descent
            self.optimizer.zero_grad()
            total_loss.backward()
            self.optimizer.step()

        # Update policy_old dengan bobot terbaru
        self.policy_old.load_state_dict(self.policy.state_dict())

        return loss_actor.item(), loss_critic.item(), loss_entropy.item()

##Setting save file model dan grafik

In [10]:
save_path = r"D:\tugas akhir\Skripsi_JSSP_Model"

# Buat foldernya otomatis jika belum ada di Drive D
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Folder baru dibuat di: {save_path}")
else:
    print(f"Folder sudah ada, hasil training akan disimpan di: {save_path}")

Folder baru dibuat di: D:\tugas akhir\Skripsi_JSSP_Model


##Random JSSP Generator

In [11]:
def paper_size_strategy(rng):
    return range_size_selector(
        rng,
        num_jobs_range=(5, 9),
        num_machines_range=(5, 9),
        allow_less_jobs_than_machines=False
    )

machine_logic = get_default_machine_matrix_creator(
    size_selector=paper_size_strategy,
    with_recirculation=False
)

duration_logic = get_default_duration_matrix_creator((1, 99))

#bagian ini yang digunakan
training_generator = modular_instance_generator(
    machine_matrix_creator=machine_logic,
    duration_matrix_creator=duration_logic,
    seed=6182201089
)

##Solver or tool

In [12]:
def solve_with_ortools(instance):
    model = cp_model.CpModel()
    n = instance.num_jobs
    m = instance.num_machines
    times = instance.duration_matrix
    machines = instance.machines_matrix
    horizon = int(np.sum(times))
    all_tasks = {}
    machine_to_intervals = [[] for _ in range(m)]

    for i in range(n):
        last_end_var = 0
        for j in range(m):
            duration = int(times[i][j])
            machine = int(machines[i][j])
            suffix = f'_{i}_{j}'
            start_var = model.NewIntVar(0, horizon, 'start' + suffix)
            end_var = model.NewIntVar(0, horizon, 'end' + suffix)
            interval_var = model.NewIntervalVar(start_var, duration, end_var, 'interval' + suffix)

            if j > 0:
                model.Add(start_var >= last_end_var)

            last_end_var = end_var
            machine_to_intervals[machine].append(interval_var)
            all_tasks[i, j] = end_var

    for i in range(m):
        model.AddNoOverlap(machine_to_intervals[i])

    makespan = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(makespan, [all_tasks[i, m - 1] for i in range(n)])
    model.Minimize(makespan)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10.0
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        return solver.Value(makespan)
    return horizon

##Validate

In [13]:
def validate(ppo_agent, env, validation_instances, val_optimal_makespans, device):
    """
    Menghitung rata-rata gap antara hasil agent dengan hasil optimal OR-Tools
    menggunakan torch.no_grad() untuk efisiensi memori.
    """
    # Pastikan model dalam mode evaluasi (mematikan dropout/batchnorm jika ada)
    ppo_agent.policy.eval() # Buka komen ini jika class PPO kamu punya atribut policy

    total_gap = 0

    with torch.no_grad():  # Mematikan perhitungan gradien
        for i, instance in enumerate(validation_instances):
            state, _ = env.reset(instance=instance)
            done = False

            while not done:
                state = state.to(device)
                legal_actions = env.get_legal_actions()

                # Memilih aksi secara deterministik/greedy biasanya lebih baik untuk validasi
                # Jika select_action kamu punya parameter 'eval_mode', set ke True
                action, _, _, _ = ppo_agent.select_action(state, legal_actions)
                state, _, done, _, _ = env.step(action)

            agent_makespan = env.current_time
            # print("__________________________________________________")
            # print(f"Agent Makespan: {agent_makespan}")
            # print(f"Optimal Makespan: {val_optimal_makespans[i]}")
            optimal = val_optimal_makespans[i]

            gap = ((agent_makespan - optimal) / optimal) * 100
            # print(f"Gap: {gap:.2f}%")
            total_gap += gap
            # print("__________________________________________________")

    # Kembalikan ke mode training setelah validasi selesai
    # ppo_agent.policy.train()

    return total_gap / len(validation_instances)

##Train

In [14]:
def train(ppo_agent, env, buffer, device, max_updates=1000):
    VALIDATION_SIZE = 20
    N_EPISODES_PER_UPDATE = 20
    REGEN_EVERY_UPDATES = 5
    log_dir = os.path.join(save_path, 'runs')
    writer = SummaryWriter(log_dir=log_dir)
    SAVE_FREQ = 50
    ema_reward = None
    smoothing_factor = 0.9
    best_gap = float('inf')

    print("Menyiapkan Validation Set menggunakan Google OR-Tools CP-SAT Solver...")
    val_instances = env.generate_instance(VALIDATION_SIZE)
    val_optimal_makespans = [solve_with_ortools(inst) for inst in val_instances]
    print(f"Selesai! Rata-rata Optimal Makespan Validasi: {np.mean(val_optimal_makespans):.2f}")

    current_batch_instances = []

    for update_iter in range(max_updates):
        if update_iter % REGEN_EVERY_UPDATES == 0:
            current_batch_instances = env.generate_instance(N_EPISODES_PER_UPDATE)
            print(f"\n--- Update {update_iter}: 20 Instance Training Baru Diregenerasi ---")

        reward_total = 0
        for i in range(N_EPISODES_PER_UPDATE):
            state, _ = env.reset(instance=current_batch_instances[i])
            done = False

            while not done:
                state = state.to(device)
                legal_actions = env.get_legal_actions()
                action, log_prob, val, mask = ppo_agent.select_action(state, legal_actions)
                next_state, reward, done, _, _ = env.step(action)
                #debugging
                # print(f"Time: {env.current_time}, Action: {action}, Reward: {reward}, done: {done}")
                # Sesuai urutan RolloutBuffer: state, action, log_prob, reward, value, is_terminal, legal_mask
                buffer.push(state, action, log_prob, reward, val, done, mask)
                state = next_state
                reward_total+= reward
            # print(f"FINAL MAKESPAN: {env.current_time}")

        with torch.no_grad():
          mean_reward = reward_total / N_EPISODES_PER_UPDATE
          writer.add_scalar("Reward/Episode", mean_reward, update_iter)
          if ema_reward is None:
            ema_reward = mean_reward
          else:
            ema_reward = smoothing_factor * ema_reward + (1 - smoothing_factor) * mean_reward

        actor,critic,entropy = ppo_agent.update(buffer)
        buffer.clear()

        # Jalankan validasi setiap selesai update
        avg_gap = validate(ppo_agent, env, val_instances, val_optimal_makespans, device)

        with torch.no_grad():
          writer.add_scalar("Validation/Gap", avg_gap, update_iter)
          writer.add_scalar("Validation/EMA Reward", ema_reward, update_iter)
          writer.add_scalar("PPO/Actor Loss", actor, update_iter)
          writer.add_scalar("PPO/Critic Loss", critic, update_iter)
          writer.add_scalar("PPO/Entropy Loss", entropy, update_iter)

          if(avg_gap < best_gap):
            best_gap = avg_gap
            checkpoint = {
              'update': update_iter,
              'model_state_dict': ppo_agent.policy.state_dict(),
              'optimizer_state_dict': ppo_agent.optimizer.state_dict(),
              'gap': avg_gap,
              'reward': mean_reward
            }
            file_name = f"{save_path}/best_model_gnn_jssp.pth"
            torch.save(checkpoint, file_name)
            print(f"--> Model Terbaik Disimpan ke Drive! Gap: {avg_gap:.2f}%")

          if update_iter % SAVE_FREQ == 0:
            periodic_file = f"{save_path}/checkpoint_step_{update_iter}.pth"
            torch.save({
                'update': update_iter,
                'model_state_dict': ppo_agent.policy.state_dict(),
                'optimizer_state_dict': ppo_agent.optimizer.state_dict(),
                'gap': avg_gap
            }, periodic_file)
            print(f"--> Checkpoint berkala disimpan di step {update_iter}")
          
          last_file = f"{save_path}/last_model_gnn_jssp.pth"
        torch.save({
            'update': update_iter,
            'model_state_dict': ppo_agent.policy.state_dict(),
            'gap': avg_gap
        }, last_file)

        print(f"Update {update_iter} selesai | Validation Gap: {avg_gap:.2f}% | Actor: {actor:.4f} | Critic: {critic:.4f}")



##Main Program

In [15]:
# ==========================================
# 3. BLOK EKSEKUSI (MAIN)
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Menggunakan Device: {device}")

    # 1. Bungkus generator milikmu menjadi Object generator
    # final_generator = wrap_generator_to_object(training_generator)
    final_generator = training_generator

    # 2. Masukkan ke dalam Environment
    env = JSSPEnv(generator=final_generator)

    # 3. Inisialisasi Model & Agen
    model = JSSPGNN(num_node_features=8).to(device)
    ppo_agent = PPO(model,device=device, lr=2.5e-4, gamma=1.0)
    buffer = RolloutBuffer()

    # 4. Mulai Training
    print("\nMemulai proses training PPO-GNN...")
    try:
        train(ppo_agent, env, buffer, device, max_updates=1000)
    except KeyboardInterrupt:
        print("\nTraining dihentikan manual. Menyimpan model...")
        torch.save(model.state_dict(), "ppo_jssp_last.pth")

Menggunakan Device: cuda

Memulai proses training PPO-GNN...
Menyiapkan Validation Set menggunakan Google OR-Tools CP-SAT Solver...
Selesai! Rata-rata Optimal Makespan Validasi: 529.05

--- Update 0: 20 Instance Training Baru Diregenerasi ---
--> Model Terbaik Disimpan ke Drive! Gap: 24.27%
--> Checkpoint berkala disimpan di step 0
Update 0 selesai | Validation Gap: 24.27% | Actor: 0.2851 | Critic: 2425.9646
--> Model Terbaik Disimpan ke Drive! Gap: 21.11%
Update 1 selesai | Validation Gap: 21.11% | Actor: 0.3457 | Critic: 1932.3690
Update 2 selesai | Validation Gap: 22.90% | Actor: 0.3451 | Critic: 2158.0242
Update 3 selesai | Validation Gap: 24.01% | Actor: 0.3165 | Critic: 1648.8292
Update 4 selesai | Validation Gap: 22.98% | Actor: 0.3265 | Critic: 1248.9756

--- Update 5: 20 Instance Training Baru Diregenerasi ---
--> Model Terbaik Disimpan ke Drive! Gap: 21.03%
Update 5 selesai | Validation Gap: 21.03% | Actor: 0.3349 | Critic: 1162.6671
--> Model Terbaik Disimpan ke Drive! Gap: 

In [16]:
import os
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# 1. Tentukan folder log (pastikan path benar)
log_dir = r"D:\tugas akhir\Skripsi_JSSP_Model\runs"

# 2. Tentukan folder tujuan hasil extract CSV
output_csv_dir = r"D:\tugas akhir\hasil_ekstraksi_csv"
if not os.path.exists(output_csv_dir):
    os.makedirs(output_csv_dir)

def extract_tensorboard_data(log_dir):
    if not os.path.exists(log_dir):
        print(f"Error: Folder {log_dir} tidak ditemukan!")
        return

    # os.walk akan menelusuri semua subfolder di dalam 'runs'
    for root, dirs, files in os.walk(log_dir):
        for file in files:
            if "tfevents" in file:
                path = os.path.join(root, file)
                
                # Ambil nama folder tepat di atas file tfevents (nama eksperimen)
                # Contoh: dari 'runs/PPO_GNN_1/events...', ambil 'PPO_GNN_1'
                experiment_name = os.path.basename(root) 
                
                print(f"\nMemproses eksperimen: {experiment_name}")
                print(f"Path: {path}")

                try:
                    event_acc = EventAccumulator(path)
                    event_acc.Reload()

                    for tag in event_acc.Tags()['scalars']:
                        scalar_events = event_acc.Scalars(tag)
                        
                        # Menambahkan kolom Wall_Time, Step, dan Value
                        df = pd.DataFrame([(e.wall_time, e.step, e.value) for e in scalar_events], 
                                          columns=['Wall_Time', 'Step', 'Value'])

                        # Membersihkan nama tag untuk jadi nama file
                        clean_tag = tag.replace("/", "_").replace("\\", "_")
                        
                        # PERBAIKAN: Nama file sekarang menggabungkan Nama Eksperimen + Nama Tag
                        # Contoh: PPO_GNN_1_Validation_Gap.csv
                        csv_name = f"{experiment_name}_{clean_tag}.csv"
                        full_csv_path = os.path.join(output_csv_dir, csv_name)
                        
                        df.to_csv(full_csv_path, index=False)
                        print(f"--- Berhasil: {csv_name}")
                except Exception as e:
                    print(f"--- Gagal memproses {file}: {e}")

# Jalankan fungsi
extract_tensorboard_data(log_dir)
print("\nSemua data telah diekstrak ke:", output_csv_dir)
                    


Memproses eksperimen: runs
Path: D:\tugas akhir\Skripsi_JSSP_Model\runs\events.out.tfevents.1778913483.DESKTOP-UD5LPJ3.5420.0
--- Berhasil: runs_Reward_Episode.csv
--- Berhasil: runs_Validation_Gap.csv
--- Berhasil: runs_Validation_EMA Reward.csv
--- Berhasil: runs_PPO_Actor Loss.csv
--- Berhasil: runs_PPO_Critic Loss.csv
--- Berhasil: runs_PPO_Entropy Loss.csv

Semua data telah diekstrak ke: D:\tugas akhir\hasil_ekstraksi_csv
